<a href="https://colab.research.google.com/github/amcclarence/McClarence_A_CIS260_IT_Risk_Project/blob/main/McClarence_Asha_CIS260_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the Socrata Open Data API client library silently
!pip install sodapy -q

In [2]:
# Import pandas for data manipulation
import pandas as pd

In [3]:
# Import the Socrata client from sodapy
from sodapy import Socrata

In [4]:
# Initialize unauthenticated public Socrata client for Chicago's data portal
client = Socrata("data.cityofchicago.org", None)

In [5]:
# Initialize unauthenticated public Socrata client for Chicago's data portal
client = Socrata("data.cityofchicago.org", None)

In [6]:
# Query the Chicago Crime dataset (dataset ID: ijzp-q8t2) for records from 2020 onwards
results = client.get("ijzp-q8t2", where="year >= 2020", limit=2000000)

In [8]:
import os

# Convert the results list to a pandas DataFrame
df_crimes = pd.DataFrame.from_records(results)

# Ensure the Project_Datasets directory exists
os.makedirs("Project_Datasets", exist_ok=True)

# Save the acquired DataFrame to disk as a CSV file in Project_Datasets
df_crimes.to_csv("Project_Datasets/crimes_2020_2026.csv", index=False)

In [9]:
# Print confirmation of loaded rows and columns
print(f"Success! Loaded {df_crimes.shape[0]:,} rows and {df_crimes.shape[1]} columns.")
print("Dataset saved to: Project_Datasets/crimes_2020_2026.csv")

Success! Loaded 1,613,963 rows and 22 columns.
Dataset saved to: Project_Datasets/crimes_2020_2026.csv


In [10]:
# Import operating system module to manage filesystem paths
import os

# Import pandas for tabular data analysis
import pandas as pd

# Import numpy for numerical and array operations
import numpy as np

In [12]:
# Define the target folder path
folder = "Project_Datasets"

# Ensure the folder exists on disk
os.makedirs(folder, exist_ok=True)

# Define path for 311 street light all-out historical CSV
path_circuit = f"{folder}/311_Service_Requests_-_Street_Lights_-_All_Out_-_Historical_20260918.csv"

# Define path for 311 alley lights out historical CSV
path_alley = f"{folder}/311_Service_Requests_-_Alley_Lights_Out_-_Historical_20260918.csv"

# Define path for Census ACS demographic ward CSV
path_acs = f"{folder}/ACS_5_Year_Data_by_Ward_20260918.csv"

# Define path for Chicago Police crimes CSV
path_crimes = f"{folder}/crimes_2020_2026.csv"

# Define path for Chicago Police traffic crashes CSV
path_crashes = f"{folder}/Traffic_Crashes_-_Crashes_20260918.csv"

# Print initialization confirmation
print("Cell 1 Complete: Environment initialized and paths defined.")

Cell 1 Complete: Environment initialized and paths defined.


In [21]:
# Define vectorized cleaning and Mean Time to Repair calculation function
def clean_and_calc_mttr(df):
    # Standardize column headers to lowercase and strip whitespace
    df.columns = df.columns.str.strip().str.lower()

    # Identify creation date column name variant
    create_col = 'creation date' if 'creation date' in df.columns else 'creation_date'

    # Identify completion date column name variant
    comp_col = 'completion date' if 'completion date' in df.columns else 'completion_date'

    # Vectorized conversion: parse creation date to datetime format, coercing errors to NaT
    df['create_dt'] = pd.to_datetime(df[create_col], errors='coerce')

    # Vectorized conversion: parse completion date to datetime format, coercing errors to NaT
    df['comp_dt'] = pd.to_datetime(df[comp_col], errors='coerce')

    # Vectorized boolean mask: completed status and non-negative repair duration
    valid_mask = (df['status'].str.lower() == 'completed') & (df['comp_dt'] >= df['create_dt'])

    # Filter valid rows and create a clean copy
    clean_df = df[valid_mask].copy()

    # Vectorized subtraction: compute duration in total seconds
    duration_sec = (clean_df['comp_dt'] - clean_df['create_dt']).dt.total_seconds()

    # Vectorized broadcasting: convert seconds to days
    clean_df['mttr_days'] = duration_sec / 86400.0

    # Vectorized broadcasting: convert seconds to hours
    clean_df['mttr_hours'] = duration_sec / 3600.0

    # Return cleaned DataFrame
    return clean_df

In [22]:
# Read raw circuit outage CSV
df_circuit = pd.read_csv(path_circuit, engine='python', on_bad_lines='skip')

# Read raw alley outage CSV
df_alley = pd.read_csv(path_alley, engine='python', on_bad_lines='skip')

# Apply vectorized hygiene and duration logic to circuit dataset
df_circuit_clean = clean_and_calc_mttr(df_circuit)

# Apply vectorized hygiene and duration logic to alley dataset
df_alley_clean = clean_and_calc_mttr(df_alley)

# Count duplicate ticket labels across the status column
dup_count = df_circuit['status'].str.contains('Duplicate', case=False, na=False).sum()

# Compute duplicate noise percentage
dup_rate = (dup_count / len(df_circuit)) * 100

In [23]:
# Print total evaluated circuit outage row count
print(f"Circuit Outages: {len(df_circuit):,} rows")

# Print total evaluated alley outage row count
print(f"Alley Outages:   {len(df_alley):,} rows")

# Print combined total public lighting work order count
print(f"Combined Lighting Records:  {len(df_circuit) + len(df_alley):,} rows")

# Print circuit Mean Time to Repair in both days and hours
print(f"Circuit All-Out MTTR: {df_circuit_clean['mttr_days'].mean():.2f} days ({df_circuit_clean['mttr_hours'].mean():.1f} hours)")

# Print alley Mean Time to Repair in both days and hours
print(f"Alley Fixture MTTR:   {df_alley_clean['mttr_days'].mean():.2f} days ({df_alley_clean['mttr_hours'].mean():.1f} hours)")

# Print duplicate ticket noise rate
print(f"Duplicate Noise Rate: {dup_rate:.1f}%")

Circuit Outages: 296,673 rows
Alley Outages:   219,010 rows
Combined Lighting Records:  515,683 rows
Circuit All-Out MTTR: 4.16 days (99.9 hours)
Alley Fixture MTTR:   49.43 days (1186.4 hours)
Duplicate Noise Rate: 0.0%


In [24]:
# Load Census American Community Survey demographic data
df_acs = pd.read_csv(path_acs, low_memory=False)

# Standardize column headers
df_acs.columns = df_acs.columns.str.strip().str.lower()

# Vectorized aggregation: calculate mean MTTR per ward
ward_mttr = df_circuit_clean.groupby('ward')['mttr_days'].mean().reset_index()

# Convert MTTR ward column to standardized string
ward_mttr['ward'] = ward_mttr['ward'].astype(str).str.split('.').str[0]

# Convert Census ward column to standardized string
df_acs['ward'] = df_acs['ward'].astype(str).str.split('.').str[0]

# Find poverty metric column
pov_cols = [c for c in df_acs.columns if 'under' in c or '25,000' in c or 'poverty' in c]

# Select primary poverty column
pov_col = pov_cols[0]

# Find total households column if present
total_hh_col = [c for c in df_acs.columns if 'total' in c and 'household' in c]

# Clean poverty metric string of currency symbols and convert to float
df_acs['poverty_metric'] = pd.to_numeric(df_acs[pov_col].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')

In [25]:
# Calculate percentage poverty rate per ward
if total_hh_col:
    # Clean total households column
    df_acs['total_hh'] = pd.to_numeric(df_acs[total_hh_col[0]].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')
    # Vectorized division: calculate poverty percentage
    df_acs['poverty_rate'] = (df_acs['poverty_metric'] / df_acs['total_hh']) * 100.0
else:
    # Use direct poverty metric if household total is omitted
    df_acs['poverty_rate'] = df_acs['poverty_metric']

In [26]:
# Merge ward MTTR performance with Census poverty demographics on ward ID
merged_ward = pd.merge(ward_mttr, df_acs[['ward', 'poverty_rate']], on='ward').dropna()

# Extract mean repair duration for Downtown Ward 42
w42 = merged_ward[merged_ward['ward'] == '42']['mttr_days'].values[0]

# Extract mean repair duration for West Side Ward 28
w28 = merged_ward[merged_ward['ward'] == '28']['mttr_days'].values[0]

# Vectorized statistical correlation: compute Pearson r
r_val = merged_ward['mttr_days'].corr(merged_ward['poverty_rate'])

In [27]:
# Print Ward 42 mean repair duration
print(f"Ward 42 (Downtown) Mean MTTR:  {w42:.2f} Days")

# Print Ward 28 mean repair duration
print(f"Ward 28 (West Side) Mean MTTR: {w28:.2f} Days")

# Print Pearson correlation coefficient
print(f"Pearson Correlation (r):       {r_val:.2f}")

Ward 42 (Downtown) Mean MTTR:  5.64 Days
Ward 28 (West Side) Mean MTTR: 4.66 Days
Pearson Correlation (r):       0.48


In [28]:
# Convert year column to numeric format
df_crimes['year'] = pd.to_numeric(df_crimes['year'], errors='coerce')

# Vectorized slice: filter records between 2020 and 2026
df_crimes_2020_2026 = df_crimes[(df_crimes['year'] >= 2020) & (df_crimes['year'] <= 2026)].copy()

# Cast boolean flags to optimize RAM
df_crimes_2020_2026['arrest'] = df_crimes_2020_2026['arrest'].astype(bool)
df_crimes_2020_2026['domestic'] = df_crimes_2020_2026['domestic'].astype(bool)

# Dynamically locate crime type column
type_col = 'primary_type' if 'primary_type' in df_crimes_2020_2026.columns else 'primary type'

# Dynamically locate location description column
loc_col = 'location_description' if 'location_description' in df_crimes_2020_2026.columns else 'location description'

# Define violent index crime classifications
violent_types = ['ASSAULT', 'BATTERY', 'ROBBERY', 'HOMICIDE']

# Vectorized filter for violent crime records
df_violent_2020_2026 = df_crimes_2020_2026[df_crimes_2020_2026[type_col].str.upper().isin(violent_types)].copy()

# Define outdoor corridor locations
corridors = ['STREET', 'SIDEWALK', 'ALLEY', 'PARKWAY', 'HIGHWAY']

# Vectorized membership test for outdoor corridors
is_corridor = df_violent_2020_2026[loc_col].str.upper().isin(corridors)

In [29]:
# Count total violent offenses
total_violent = len(df_violent_2020_2026)

# Vectorized sum of outdoor violent offenses
outdoor_violent = is_corridor.sum()

# Compute ratio of corridor violence to total violence
ratio = (outdoor_violent / total_violent) * 100.0 if total_violent > 0 else 0.0

In [30]:
# Print total evaluated crime records
print(f"Total 2020-2026 Crime Records:           {len(df_crimes_2020_2026):,}")

# Print total evaluated violent offenses
print(f"Total 2020-2026 Violent Offenses:        {total_violent:,}")

# Print count of violent incidents in outdoor corridors
print(f"2020-2026 Outdoor Corridor Incidents:    {outdoor_violent:,}")

# Print percentage ratio of corridor violence
print(f"2020-2026 Corridor Violence Ratio:       {ratio:.1f}%")

Total 2020-2026 Crime Records:           1,613,963
Total 2020-2026 Violent Offenses:        494,288
2020-2026 Outdoor Corridor Incidents:    157,455
2020-2026 Corridor Violence Ratio:       31.9%


In [31]:
# Read traffic crashes CSV file into memory
df_crashes = pd.read_csv(path_crashes, engine='python', on_bad_lines='skip')

# Standardize column headers to lowercase
df_crashes.columns = df_crashes.columns.str.strip().str.lower()

# Dynamically locate crash date column
date_col = 'crash_date' if 'crash_date' in df_crashes.columns else 'crash date'

# Vectorized conversion: parse crash timestamps to datetime
df_crashes['crash_dt'] = pd.to_datetime(df_crashes[date_col], errors='coerce')

# Vectorized extraction of calendar year
df_crashes['crash_year'] = df_crashes['crash_dt'].dt.year

# Vectorized boolean mask for 2020-2026 window
mask_crashes_period = (df_crashes['crash_year'] >= 2020) & (df_crashes['crash_year'] <= 2026)

# Filter crash dataframe
df_crashes_period = df_crashes[mask_crashes_period].copy()

# Dynamically locate lighting condition column
light_col = 'lighting_condition' if 'lighting_condition' in df_crashes_period.columns else 'lighting condition'

# Dynamically locate total injuries column
injury_col = 'injuries_total' if 'injuries_total' in df_crashes_period.columns else 'injuries total'

# Vectorized comparison: isolate crashes coded as total unlit darkness
is_unlit_darkness = df_crashes_period[light_col].str.upper().str.strip() == 'DARKNESS'

# Slice dataframe to retain only crashes under unlit darkness
df_dark_crashes = df_crashes_period[is_unlit_darkness].copy()

/tmp/ipykernel_1042/4033311368.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_crashes['crash_dt'] = pd.to_datetime(df_crashes[date_col], errors='coerce')


In [32]:
# Count total crashes evaluated across the period
total_period_crashes = len(df_crashes_period)

# Count crashes occurring specifically in pitch darkness
total_dark_crashes = len(df_dark_crashes)

In [33]:
# Calculate percentage of total crashes that occurred in darkness
dark_crash_ratio = (total_dark_crashes / total_period_crashes) * 100.0 if total_period_crashes > 0 else 0.0

In [34]:
# Vectorized conversion: parse injuries column to numeric, filling NaNs with 0
df_dark_crashes[injury_col] = pd.to_numeric(df_dark_crashes[injury_col], errors='coerce').fillna(0)

# Vectorized sum: compute total physical injuries sustained in darkness crashes
total_dark_injuries = int(df_dark_crashes[injury_col].sum())

In [35]:
# Print total evaluated traffic collisions
print(f"Total 2020-2026 Evaluated Crashes:      {total_period_crashes:,}")

# Print count of crashes occurring in unlit darkness
print(f"Crashes in Unlit Darkness ('DARKNESS'):  {total_dark_crashes:,}")

# Print ratio of darkness crashes
print(f"Unlit Darkness Crash Ratio:             {dark_crash_ratio:.2f}%")

# Print total documented injuries sustained in unlit crashes
print(f"Total Injuries in Unlit Crashes:        {total_dark_injuries:,}")

Total 2020-2026 Evaluated Crashes:      720,642
Crashes in Unlit Darkness ('DARKNESS'):  31,371
Unlit Darkness Crash Ratio:             4.35%
Total Injuries in Unlit Crashes:        6,753


In [36]:
# Set smart pole count managed per neighborhood field gateway collector
poles = 500

# Set hardware replacement cost per smart LED luminaire unit
unit_cost = 350.0

# Compute total capital asset value for a 500-pole gateway cluster
asset_value = poles * unit_cost

# Set exposure factor reflecting emergency response, towing, and injury liability
exposure_factor = 0.45

In [37]:
# Calculate Single Loss Expectancy: dollar loss experienced per major blackout
sle = asset_value * exposure_factor

In [38]:
# Set Inherent Annual Rate of Occurrence: unmitigated baseline cascading blackouts/yr
aro_inherent = 14

In [39]:
# Calculate Inherent Annual Loss Expectancy: unmitigated annual financial risk
ale_inherent = sle * aro_inherent

In [40]:
# Set Residual Annual Rate of Occurrence: blackouts/yr with automated watchdogs
aro_residual = 2

In [41]:
# Calculate Residual Annual Loss Expectancy: mitigated annual financial risk
ale_residual = sle * aro_residual

In [42]:
# Calculate percentage net risk reduction achieved by technical controls
reduction_pct = ((ale_inherent - ale_residual) / ale_inherent) * 100.0

In [43]:
# Print calculated Single Loss Expectancy per blackout incident
print(f"Single Loss Expectancy (SLE):  ${sle:,.2f}")

# Print unmitigated Inherent Annual Loss Expectancy
print(f"Inherent Risk (ALE):           ${ale_inherent:,.2f}/year")

# Print mitigated Residual Annual Loss Expectancy
print(f"Residual Risk (ALE):           ${ale_residual:,.2f}/year")

# Print net enterprise risk reduction percentage
print(f"Net Risk Reduction:            {reduction_pct:.1f}%")

Single Loss Expectancy (SLE):  $78,750.00
Inherent Risk (ALE):           $1,102,500.00/year
Residual Risk (ALE):           $157,500.00/year
Net Risk Reduction:            85.7%
